In [1]:
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout)

Fri May 29 13:10:41 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 590.63                 Driver Version: 592.02         CUDA Version: 13.1     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 5070 ...    On  |   00000000:01:00.0 Off |                  N/A |
| N/A   50C    P8              4W /  115W |      21MiB /   8151MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [8]:
import json
from pathlib import Path

best_weights = {
    'variant': 1.0,
    'also_bought': 1.0,
    'item2item_cf': 1.0,
    'category_popular': 1.0,
    'purchase_interval': 1.0,
    'user_category_popular': 1.0,
    'timedecay': 2.0,
    'repurchase': 1.0,
    'bin_popular': 0.5,
    'general_pred': 0.0
}

weights_path = Path('/workspace/outputs/candidates/best_weights.json')
weights_path.parent.mkdir(parents=True, exist_ok=True)
with open(weights_path, 'w') as f:
    json.dump(best_weights, f, indent=2)
print(f"保存完了: {weights_path}")
print(f"best_recall: 0.056413")
print(f"best_weights: {best_weights}")
best_recall = 0.056413

保存完了: /workspace/outputs/candidates/best_weights.json
best_recall: 0.056413
best_weights: {'variant': 1.0, 'also_bought': 1.0, 'item2item_cf': 1.0, 'category_popular': 1.0, 'purchase_interval': 1.0, 'user_category_popular': 1.0, 'timedecay': 2.0, 'repurchase': 1.0, 'bin_popular': 0.5, 'general_pred': 0.0}


In [2]:
import polars as pl
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import date, timedelta
from collections import defaultdict

DATA_DIR = Path('/workspace/data/raw')
CANDIDATE_DIR = Path('/workspace/outputs/candidates')
CANDIDATE_DIR.mkdir(parents=True, exist_ok=True)

# a, b, c, d = 2.5e4, 1.5e5, 2e-1, 1e3
TD_A, TD_B, TD_C, TD_D = 2.5e4, 1.5e5, 2e-1, 1e3

# ============================================================
# データ読み込み
# ============================================================
transactions_raw = pl.read_csv(
    DATA_DIR / 'transactions_train.csv',
    schema_overrides={'article_id': pl.String, 'customer_id': pl.String}
).with_columns(pl.col('t_dat').str.to_date())

customers_raw = pl.read_csv(
    DATA_DIR / 'customers.csv',
    schema_overrides={'customer_id': pl.String}
)

articles_raw = pl.read_csv(
    DATA_DIR / 'articles.csv',
    schema_overrides={'article_id': pl.String}
)

# ============================================================
# 整数変換マップ作成
# ============================================================
print("整数変換マップ作成中...")

all_customer_ids = pl.concat([
    transactions_raw.select('customer_id'),
    customers_raw.select('customer_id')
]).unique().sort('customer_id')['customer_id'].to_list()

all_article_ids = pl.concat([
    transactions_raw.select('article_id'),
    articles_raw.select('article_id')
]).unique().sort('article_id')['article_id'].to_list()

customer_id_map = {v: i for i, v in enumerate(all_customer_ids)}
customer_id_reverse = {i: v for v, i in customer_id_map.items()}
article_id_map = {v: i for i, v in enumerate(all_article_ids)}
article_id_reverse = {i: v for v, i in article_id_map.items()}

print(f"customer数: {len(customer_id_map):,}")
print(f"article数: {len(article_id_map):,}")

# ============================================================
# 整数変換済みデータ作成
# ============================================================
print("整数変換中...")

transactions = (
    transactions_raw
    .with_columns([
        pl.col('customer_id').replace_strict(customer_id_map, default=None).cast(pl.Int32),
        pl.col('article_id').replace_strict(article_id_map, default=None).cast(pl.Int32),
    ])
    .drop_nulls()
)

customers = (
    customers_raw
    .with_columns([
        pl.col('customer_id').replace_strict(customer_id_map, default=None).cast(pl.Int32),
        pl.when(pl.col('age').is_null()).then(-1)
        .otherwise((pl.col('age') / 10).cast(pl.Int32) * 10)
        .alias('age_bin'),
        pl.col('age').fill_null(-1).cast(pl.Int32),
        pl.col('FN').fill_null(0),
        pl.col('Active').fill_null(0),
        pl.col('club_member_status').fill_null('UNKNOWN'),
        pl.col('fashion_news_frequency').fill_null('NONE'),
    ])
    .drop_nulls(subset=['customer_id'])
)

articles = (
    articles_raw
    .with_columns(
        pl.col('article_id').replace_strict(article_id_map, default=None).cast(pl.Int32)
    )
    .drop_nulls(subset=['article_id'])
)

# product_codeマッピング
product_code_map = {}
article_to_product = {}
for row in articles.select(['article_id', 'product_code']).iter_rows(named=True):
    code = row['product_code']
    art = row['article_id']
    article_to_product[art] = code
    if code not in product_code_map:
        product_code_map[code] = []
    product_code_map[code].append(art)

# カテゴリマッピング
CATEGORY_COLS = ['section_name', 'garment_group_name', 'index_group_name']
articles_category = articles_raw.with_columns(
    pl.col('article_id').replace_strict(article_id_map, default=None).cast(pl.Int32)
).drop_nulls(subset=['article_id']).select(['article_id'] + CATEGORY_COLS)

last_ts = transactions['t_dat'].max()
print(f"全データ最終日: {last_ts}")
print(f"transactions: {len(transactions):,}行")
print("整数変換完了!")

整数変換マップ作成中...
customer数: 1,371,980
article数: 105,542
整数変換中...
全データ最終日: 2020-09-22
transactions: 31,788,324行
整数変換完了!


In [3]:
# ============================================================
# 共通関数
# ============================================================
def get_actuals(transactions, cutoff, label_days=7):
    label_end = cutoff + timedelta(days=label_days)
    grouped = (
        transactions
        .filter((pl.col('t_dat') > pl.lit(cutoff)) & (pl.col('t_dat') <= pl.lit(label_end)))
        .group_by('customer_id')
        .agg(pl.col('article_id').unique().alias('actual'))
    )
    return {row['customer_id']: set(row['actual']) for row in grouped.iter_rows(named=True)}

def apk(actual_set, predicted, k):
    if not actual_set:
        return 0.0
    score = 0.0
    hits = 0
    seen = set()
    for rank, article_id in enumerate(predicted[:k], start=1):
        if article_id in seen:
            continue
        seen.add(article_id)
        if article_id in actual_set:
            hits += 1
            score += hits / rank
    return score / min(len(actual_set), k)

def save_candidates(df, name, cutoff):
    """候補をparquetに保存"""
    path = CANDIDATE_DIR / f"{name}_{cutoff}.parquet"
    df.write_parquet(path)
    print(f"  保存: {path}")
    return df

def load_candidates(name, cutoff):
    """保存済み候補を読み込み"""
    path = CANDIDATE_DIR / f"{name}_{cutoff}.parquet"
    if path.exists():
        print(f"  読み込み: {path}")
        return pl.read_parquet(path)
    return None

# ============================================================
# 候補生成ロジック
# ============================================================

def build_timedecay(transactions, cutoff, top_n=120):
    name = 'timedecay'
    cached = load_candidates(name, cutoff)
    if cached is not None:
        return cached
    
    print(f"  [{name}] 計算中...")
    df_train = transactions.filter(pl.col('t_dat') <= pl.lit(cutoff))
    df_train = df_train.with_columns(
        (pl.col('t_dat') +
         ((pl.lit(cutoff) - pl.col('t_dat')).dt.total_days() % 7
         ).cast(pl.Int32) * pl.duration(days=1)
        ).alias('ldbw')
    )
    weekly_sales = df_train.group_by(['ldbw', 'article_id']).agg(pl.len().alias('count'))
    df_train = df_train.join(weekly_sales, on=['ldbw', 'article_id'], how='left')
    last_week_sales = (
        weekly_sales.filter(pl.col('ldbw') == cutoff)
        .select(['article_id', 'count']).rename({'count': 'count_targ'})
    )
    df_train = (
        df_train
        .join(last_week_sales, on='article_id', how='left')
        .with_columns(pl.col('count_targ').fill_null(0))
        .with_columns((pl.col('count_targ') / pl.col('count')).alias('quotient'))
        .with_columns(
            (pl.lit(cutoff) - pl.col('t_dat')).dt.total_days().clip(lower_bound=1).alias('days_ago')
        )
        .with_columns(
            (
                pl.col('quotient') * (
                    TD_A / pl.col('days_ago').cast(pl.Float64).sqrt() +
                    TD_B * (-TD_C * pl.col('days_ago').cast(pl.Float64)).exp() - TD_D
                ).clip(lower_bound=0)
            ).alias('score')
        )
    )
    result = (
        df_train.group_by(['customer_id', 'article_id'])
        .agg(pl.col('score').sum())
        .filter(pl.col('score') > 0)
        .with_columns(
            pl.col('score').rank(method='ordinal', descending=True).over('customer_id').alias('rank')
        )
        .filter(pl.col('rank') <= top_n)
        .select(['customer_id', 'article_id', 'score', 'rank'])
        .with_columns([
            pl.col('customer_id').cast(pl.Int32),
            pl.col('article_id').cast(pl.Int32),
            pl.col('rank').cast(pl.Int32),
        ])
    )
    print(f"  [{name}] 候補数: {len(result):,}")
    return save_candidates(result, name, cutoff)

def build_repurchase(transactions, cutoff, top_n=120):
    """②直近購買日順候補"""
    name = 'repurchase'
    cached = load_candidates(name, cutoff)
    if cached is not None:
        return cached
    
    print(f"  [{name}] 計算中...")
    df_train = transactions.filter(pl.col('t_dat') <= pl.lit(cutoff))
    result = (
        df_train
        .group_by(['customer_id', 'article_id'])
        .agg(pl.max('t_dat').alias('last_date'))
        .with_columns(
            pl.col('last_date').rank(method='ordinal', descending=True).over('customer_id').alias('rank')
        )
        .filter(pl.col('rank') <= top_n)
        .with_columns(
            (pl.lit(cutoff) - pl.col('last_date')).dt.total_days().cast(pl.Float64).alias('score')
        )
        .with_columns(pl.col('score').map_elements(lambda x: 1.0 / (1.0 + x), return_dtype=pl.Float64))
        .select(['customer_id', 'article_id', 'score', 'rank'])
        .with_columns([
            pl.col('customer_id').cast(pl.Int32),
            pl.col('article_id').cast(pl.Int32),
            pl.col('rank').cast(pl.Int32),
        ])
    )
    print(f"  [{name}] 候補数: {len(result):,}")
    return save_candidates(result, name, cutoff)

def build_variant(transactions, cutoff, timedecay_df=None, top_n=30):
    """③同一product codeバリアント候補"""
    name = 'variant'
    cached = load_candidates(name, cutoff)
    if cached is not None:
        return cached
    
    print(f"  [{name}] 計算中...")
    
    if timedecay_df is None:
        timedecay_df = load_candidates('timedecay', cutoff)
    
    articles_product = articles.select(['article_id', 'product_code'])
    
    result = (
        timedecay_df
        .filter(pl.col('rank') <= 50)  # top50件に絞る
        .select(['customer_id', 'article_id', 'score'])
        .join(articles_product, on='article_id', how='left')
        .join(
            articles_product.rename({'article_id': 'variant_article_id'}),
            on='product_code', how='left'
        )
        .filter(pl.col('article_id') != pl.col('variant_article_id'))
        .drop_nulls(subset=['variant_article_id'])
        .group_by(['customer_id', pl.col('variant_article_id').alias('article_id')])
        .agg(pl.col('score').sum().alias('score'))
        .with_columns(
            pl.col('score').rank(method='ordinal', descending=True).over('customer_id').alias('rank')
        )
        .filter(pl.col('rank') <= top_n)
        .select(['customer_id', 'article_id', 'score', 'rank'])
        .with_columns([
            pl.col('customer_id').cast(pl.Int32),
            pl.col('article_id').cast(pl.Int32),
            pl.col('rank').cast(pl.Int32),
        ])
    )
    print(f"  [{name}] 候補数: {len(result):,}")
    return save_candidates(result, name, cutoff)

# print("関数定義完了")

def build_also_bought(transactions, cutoff, repurchase_df=None, threshold=10, top_n=20, history_days=180):
    """④also bought together候補（直近6ヶ月版）"""
    name = 'also_bought'
    cached = load_candidates(name, cutoff)
    if cached is not None:
        return cached
    
    print(f"  [{name}] 計算中（直近{history_days}日・月ごと分割）...")
    
    # 直近6ヶ月に絞る
    start_date = cutoff - timedelta(days=history_days)
    df_train = transactions.filter(
        (pl.col('t_dat') > pl.lit(start_date)) &
        (pl.col('t_dat') <= pl.lit(cutoff))
    )
    
    months = (
        df_train
        .select(pl.col('t_dat').dt.truncate('1mo').alias('month'))
        .unique().sort('month')
        .to_series().to_list()
    )
    print(f"    対象月数: {len(months)}")
    
    all_pairs = []
    for i, month in enumerate(months):
        month_end = month + timedelta(days=31)
        df_month = df_train.filter(
            (pl.col('t_dat') >= pl.lit(month)) & (pl.col('t_dat') < pl.lit(month_end))
        )
        if len(df_month) == 0:
            continue
        pairs = (
            df_month.select(['customer_id', 't_dat', 'article_id'])
            .join(df_month.select(['customer_id', 't_dat', 'article_id']), on=['customer_id', 't_dat'], how='inner')
            .filter(pl.col('article_id') != pl.col('article_id_right'))
            .group_by(['article_id', 'article_id_right'])
            .agg(pl.len().alias('count'))
        )
        all_pairs.append(pairs)
        print(f"    進捗: {i+1}/{len(months)}ヶ月")
    
    also_bought_dict = defaultdict(list)
    combined = (
        pl.concat(all_pairs)
        .group_by(['article_id', 'article_id_right'])
        .agg(pl.col('count').sum())
        .filter(pl.col('count') >= threshold)
        .sort(['article_id', 'count'], descending=[False, True])
    )
    for row in combined.iter_rows(named=True):
        also_bought_dict[row['article_id']].append((row['article_id_right'], row['count']))
    
    # ユーザーの直近1ヶ月購買履歴から候補生成
    recent_start = cutoff - timedelta(days=30)
    user_history = (
        df_train.filter(pl.col('t_dat') >= pl.lit(recent_start))
        .sort(['customer_id', 't_dat'], descending=[False, True])
        .group_by('customer_id')
        .agg(pl.col('article_id').alias('articles'))
    )
    user_history_dict = {
        row['customer_id']: row['articles']
        for row in user_history.iter_rows(named=True)
    }
    
    rows = []
    for customer_id, hist_articles in user_history_dict.items():
        purchased_set = set(hist_articles)
        candidates = {}
        for art in hist_articles:
            for paired_art, count in also_bought_dict.get(art, []):
                if paired_art not in purchased_set:
                    candidates[paired_art] = candidates.get(paired_art, 0) + count
        
        sorted_cands = sorted(candidates.items(), key=lambda x: -x[1])[:top_n]
        for rank, (art, score) in enumerate(sorted_cands, start=1):
            rows.append({
                'customer_id': customer_id,
                'article_id': art,
                'score': float(score),
                'rank': rank,
            })
    
    result = pl.DataFrame(rows).with_columns([
        pl.col('customer_id').cast(pl.Int32),
        pl.col('article_id').cast(pl.Int32),
        pl.col('rank').cast(pl.Int32),
    ])
    print(f"  [{name}] 候補数: {len(result):,}")
    return save_candidates(result, name, cutoff)


def build_item2item_cf(transactions, cutoff, top_n=20):
    """⑤item2item CF候補（repurchase_df引数を削除）"""
    name = 'item2item_cf'
    cached = load_candidates(name, cutoff)
    if cached is not None:
        return cached
    
    print(f"  [{name}] 計算中...")
    from scipy.sparse import csr_matrix
    from sklearn.preprocessing import normalize
    
    df_train = transactions.filter(pl.col('t_dat') <= pl.lit(cutoff))
    
    user_item = (
        df_train
        .group_by(['customer_id', 'article_id'])
        .agg(pl.len().alias('count'))
    )
    
    n_users = len(customer_id_map)
    n_items = len(article_id_map)
    
    row = user_item['customer_id'].to_numpy()
    col = user_item['article_id'].to_numpy()
    data = user_item['count'].to_numpy().astype(np.float32)
    
    uim = csr_matrix((data, (row, col)), shape=(n_users, n_items))
    iim = uim.T.tocsr()
    iim_norm = normalize(iim, norm='l2')
    
    print(f"    行列サイズ: {iim_norm.shape}")
    
    start_date = cutoff - timedelta(days=30)
    recent_articles = (
        df_train.filter(pl.col('t_dat') >= pl.lit(start_date))
        .select('article_id').unique()
        .to_series().to_list()
    )
    print(f"    直近1ヶ月の購買商品数: {len(recent_articles):,}")
    
    print("    類似度辞書を事前計算中...")
    similar_items_dict = {}
    for i, art in enumerate(recent_articles):
        if i % 10000 == 0:
            print(f"      進捗: {i:,} / {len(recent_articles):,}")
        if art >= iim_norm.shape[0]:
            continue
        sim_scores = iim_norm[art].dot(iim_norm.T).toarray().flatten()
        top_items = np.argsort(sim_scores)[::-1][1:top_n+1]
        similar_items_dict[art] = [
            (int(idx), float(sim_scores[idx]))
            for idx in top_items
            if sim_scores[idx] > 0
        ]
    
    print(f"    類似度辞書完成: {len(similar_items_dict):,}商品")
    
    recent_hist = (
        df_train.filter(pl.col('t_dat') >= pl.lit(start_date))
        .group_by('customer_id')
        .agg(pl.col('article_id').unique().alias('articles'))
    )
    
    rows_list = []
    for i, row_data in enumerate(recent_hist.iter_rows(named=True)):
        if i % 100000 == 0:
            print(f"      進捗: {i:,} / {len(recent_hist):,}")
        
        cid = row_data['customer_id']
        hist_arts = row_data['articles']
        purchased_set = set(hist_arts)
        
        candidates = {}
        for art in hist_arts:
            for sim_art, sim_score in similar_items_dict.get(art, []):
                if sim_art not in purchased_set:
                    candidates[sim_art] = candidates.get(sim_art, 0) + sim_score
        
        sorted_cands = sorted(candidates.items(), key=lambda x: -x[1])[:top_n]
        for rank, (art, score) in enumerate(sorted_cands, start=1):
            rows_list.append({
                'customer_id': cid,
                'article_id': art,
                'score': score,
                'rank': rank,
            })
    
    result = pl.DataFrame(rows_list).with_columns([
        pl.col('customer_id').cast(pl.Int32),
        pl.col('article_id').cast(pl.Int32),
        pl.col('rank').cast(pl.Int32),
    ])
    print(f"  [{name}] 候補数: {len(result):,}")
    return save_candidates(result, name, cutoff)

print("関数定義完了")

# print("関数定義完了")

def build_category_popular(transactions, cutoff, top_n=20):
    """⑥同カテゴリ内人気商品候補"""
    name = 'category_popular'
    cached = load_candidates(name, cutoff)
    if cached is not None:
        return cached
    
    print(f"  [{name}] 計算中...")
    df_train = transactions.filter(pl.col('t_dat') <= pl.lit(cutoff))
    start_date = cutoff - timedelta(days=30)
    recent = df_train.filter(pl.col('t_dat') >= pl.lit(start_date))
    
    # カテゴリ別人気商品
    category_top = {}
    for col in CATEGORY_COLS:
        recent_with_cat = recent.join(
            articles_category.select(['article_id', col]),
            on='article_id', how='left'
        )
        counts = (
            recent_with_cat.filter(pl.col(col).is_not_null())
            .group_by([col, 'article_id'])
            .agg(pl.len().alias('count'))
            .sort([col, 'count'], descending=[False, True])
        )
        top_for_col = defaultdict(list)
        for row in counts.iter_rows(named=True):
            cat = row[col]
            if len(top_for_col[cat]) < top_n:
                top_for_col[cat].append((row['article_id'], row['count']))
        category_top[col] = dict(top_for_col)
    
    # ユーザーの好みカテゴリを特定
    history_with_cat = (
        df_train.filter(pl.col('t_dat') >= pl.lit(cutoff - timedelta(days=90)))
        .join(articles_category, on='article_id', how='left')
    )
    
    rows_list = []
    for col in CATEGORY_COLS:
        user_cat_counts = (
            history_with_cat.filter(pl.col(col).is_not_null())
            .group_by(['customer_id', col])
            .agg(pl.len().alias('count'))
            .sort(['customer_id', 'count'], descending=[False, True])
        )
        user_top_cats = defaultdict(list)
        for row in user_cat_counts.iter_rows(named=True):
            cid = row['customer_id']
            if len(user_top_cats[cid]) < 2:
                user_top_cats[cid].append(row[col])
        
        purchased = (
            df_train.group_by(['customer_id', 'article_id'])
            .agg(pl.len())
            .group_by('customer_id')
            .agg(pl.col('article_id').alias('articles'))
        )
        purchased_dict = {
            row['customer_id']: set(row['articles'])
            for row in purchased.iter_rows(named=True)
        }
        
        for cid, cats in user_top_cats.items():
            purchased_set = purchased_dict.get(cid, set())
            candidates = {}
            for cat in cats:
                for art, count in category_top[col].get(cat, []):
                    if art not in purchased_set:
                        candidates[art] = candidates.get(art, 0) + count
            
            sorted_cands = sorted(candidates.items(), key=lambda x: -x[1])[:top_n]
            for rank, (art, score) in enumerate(sorted_cands, start=1):
                rows_list.append({
                    'customer_id': cid,
                    'article_id': art,
                    'score': float(score),
                    'rank': rank,
                })
    
    result = (
        pl.DataFrame(rows_list)
        .group_by(['customer_id', 'article_id'])
        .agg(pl.col('score').sum(), pl.col('rank').min())
        .with_columns([
            pl.col('customer_id').cast(pl.Int32),
            pl.col('article_id').cast(pl.Int32),
            pl.col('rank').cast(pl.Int32),
        ])
    )
    print(f"  [{name}] 候補数: {len(result):,}")
    return save_candidates(result, name, cutoff)

def build_bin_popular(transactions, cutoff, top_n=50):
    """⑦年齢ビン別人気商品候補"""
    name = 'bin_popular'
    cached = load_candidates(name, cutoff)
    if cached is not None:
        return cached
    
    print(f"  [{name}] 計算中...")
    df_train = transactions.filter(pl.col('t_dat') <= pl.lit(cutoff))
    recent = (
        df_train
        .filter(pl.col('t_dat') >= pl.lit(cutoff - timedelta(days=28)))
        .join(customers.select(['customer_id', 'age_bin']), on='customer_id', how='left')
    )
    bin_popular = (
        recent
        .group_by(['age_bin', 'article_id'])
        .agg(pl.len().alias('count'))
        .with_columns(
            pl.col('count').rank(method='ordinal', descending=True).over('age_bin').alias('rank')
        )
        .filter(pl.col('rank') <= top_n)
    )
    result = (
        customers.select(['customer_id', 'age_bin'])
        .join(bin_popular.select(['age_bin', 'article_id', 'count', 'rank']), on='age_bin', how='left')
        .drop_nulls()
        .with_columns(pl.col('count').cast(pl.Float64).alias('score'))
        .select(['customer_id', 'article_id', 'score', 'rank'])
        .with_columns([
            pl.col('customer_id').cast(pl.Int32),
            pl.col('article_id').cast(pl.Int32),
            pl.col('rank').cast(pl.Int32),
        ])
    )
    print(f"  [{name}] 候補数: {len(result):,}")
    return save_candidates(result, name, cutoff)

def build_general_pred(transactions, cutoff, top_n=50):
    """⑧全体トレンド候補（修正版）"""
    name = 'general_pred'
    cached = load_candidates(name, cutoff)
    if cached is not None:
        return cached
    
    print(f"  [{name}] 計算中...")
    df_train = transactions.filter(pl.col('t_dat') <= pl.lit(cutoff))
    df_train_td = df_train.with_columns(
        (pl.col('t_dat') +
         ((pl.lit(cutoff) - pl.col('t_dat')).dt.total_days() % 7
         ).cast(pl.Int32) * pl.duration(days=1)
        ).alias('ldbw')
    )
    weekly_sales = df_train_td.group_by(['ldbw', 'article_id']).agg(pl.len().alias('count'))
    df_train_td = df_train_td.join(weekly_sales, on=['ldbw', 'article_id'], how='left')
    last_week_sales = (
        weekly_sales.filter(pl.col('ldbw') == cutoff)
        .select(['article_id', 'count']).rename({'count': 'count_targ'})
    )
    df_train_td = (
        df_train_td
        .join(last_week_sales, on='article_id', how='left')
        .with_columns(pl.col('count_targ').fill_null(0))
        .with_columns((pl.col('count_targ') / pl.col('count')).alias('quotient'))
    )
    
    # top_n件を取得
    general_arts = (
        df_train_td.group_by('article_id')
        .agg(pl.col('quotient').sum().alias('score'))
        .sort('score', descending=True)
        .head(top_n)
        .with_columns(
            pl.col('score').rank(method='ordinal', descending=True).cast(pl.Int32).alias('rank')
        )
        .select(['article_id', 'score', 'rank'])
        .with_columns([
            pl.col('article_id').cast(pl.Int32),
            pl.col('rank').cast(pl.Int32),
        ])
    )
    
    # Polarsでjoin（cross joinを避ける）
    # 全ユーザーにダミーキーでjoin
    all_customers_df = customers.select('customer_id').with_columns(
        pl.lit(1).alias('key')
    )
    general_arts_key = general_arts.with_columns(
        pl.lit(1).alias('key')
    )
    
    result = (
        all_customers_df
        .join(general_arts_key, on='key', how='inner')
        .drop('key')
        .with_columns(pl.col('customer_id').cast(pl.Int32))
    )
    
    print(f"  [{name}] 候補数: {len(result):,}")
    return save_candidates(result, name, cutoff)

def build_purchase_interval(transactions, cutoff, top_n=20):
    """⑨購買間隔ベース候補（そろそろ買い直す時期の商品）"""
    name = 'purchase_interval'
    cached = load_candidates(name, cutoff)
    if cached is not None:
        return cached
    
    print(f"  [{name}] 計算中...")
    df_train = transactions.filter(pl.col('t_dat') <= pl.lit(cutoff))
    
    # 商品ごとの平均購買間隔を計算
    purchase_dates = (
        df_train
        .sort(['customer_id', 'article_id', 't_dat'])
        .group_by(['customer_id', 'article_id'])
        .agg([
            pl.col('t_dat').sort().alias('dates'),
            pl.col('t_dat').max().alias('last_date'),
            pl.len().alias('purchase_count')
        ])
        .filter(pl.col('purchase_count') >= 2)
    )
    
    rows_list = []
    for row in purchase_dates.iter_rows(named=True):
        dates = row['dates']
        if len(dates) < 2:
            continue
        diffs = [(dates[i+1] - dates[i]).days for i in range(len(dates)-1)]
        avg_interval = sum(diffs) / len(diffs)
        days_since_last = (cutoff - row['last_date']).days
        # 購買間隔に近いほどスコアが高い
        score = 1.0 / (1.0 + abs(days_since_last - avg_interval))
        rows_list.append({
            'customer_id': row['customer_id'],
            'article_id': row['article_id'],
            'score': score,
            'avg_interval': avg_interval,
            'days_since_last': days_since_last,
        })
    
    result = (
        pl.DataFrame(rows_list)
        .with_columns(
            pl.col('score').rank(method='ordinal', descending=True).over('customer_id').alias('rank')
        )
        .filter(pl.col('rank') <= top_n)
        .select(['customer_id', 'article_id', 'score', 'rank'])
        .with_columns([
            pl.col('customer_id').cast(pl.Int32),
            pl.col('article_id').cast(pl.Int32),
            pl.col('rank').cast(pl.Int32),
        ])
    )
    print(f"  [{name}] 候補数: {len(result):,}")
    return save_candidates(result, name, cutoff)

def build_user_category_popular(transactions, cutoff, top_n=20):
    """⑩ユーザーの好みカテゴリ×カテゴリ内人気候補"""
    name = 'user_category_popular'
    cached = load_candidates(name, cutoff)
    if cached is not None:
        return cached
    
    print(f"  [{name}] 計算中...")
    df_train = transactions.filter(pl.col('t_dat') <= pl.lit(cutoff))
    start_date = cutoff - timedelta(days=30)
    recent = df_train.filter(pl.col('t_dat') >= pl.lit(start_date))
    
    # カテゴリ別人気商品（直近1ヶ月）
    recent_with_cat = recent.join(
        articles.select(['article_id', 'product_type_no', 'garment_group_no']),
        on='article_id', how='left'
    )
    product_type_popular = (
        recent_with_cat.filter(pl.col('product_type_no').is_not_null())
        .group_by(['product_type_no', 'article_id'])
        .agg(pl.len().alias('count'))
        .with_columns(
            pl.col('count').rank(method='ordinal', descending=True).over('product_type_no').alias('rank')
        )
        .filter(pl.col('rank') <= top_n)
    )
    
    # ユーザーの好みproduct_type（直近3ヶ月）
    hist_with_cat = (
        df_train.filter(pl.col('t_dat') >= pl.lit(cutoff - timedelta(days=90)))
        .join(articles.select(['article_id', 'product_type_no']), on='article_id', how='left')
        .filter(pl.col('product_type_no').is_not_null())
    )
    user_top_product_types = (
        hist_with_cat
        .group_by(['customer_id', 'product_type_no'])
        .agg(pl.len().alias('count'))
        .with_columns(
            pl.col('count').rank(method='ordinal', descending=True).over('customer_id').alias('rank')
        )
        .filter(pl.col('rank') <= 3)
        .select(['customer_id', 'product_type_no'])
    )
    
    result = (
        user_top_product_types
        .join(product_type_popular.select(['product_type_no', 'article_id', 'count', 'rank']),
              on='product_type_no', how='left')
        .drop_nulls()
        .group_by(['customer_id', 'article_id'])
        .agg(pl.col('count').sum().alias('score'), pl.col('rank').min().alias('rank'))
        .with_columns(
            pl.col('score').rank(method='ordinal', descending=True).over('customer_id').alias('rank')
        )
        .filter(pl.col('rank') <= top_n)
        .select(['customer_id', 'article_id', 'score', 'rank'])
        .with_columns([
            pl.col('customer_id').cast(pl.Int32),
            pl.col('article_id').cast(pl.Int32),
            pl.col('rank').cast(pl.Int32),
        ])
    )
    print(f"  [{name}] 候補数: {len(result):,}")
    return save_candidates(result, name, cutoff)

print("全候補生成関数定義完了")

関数定義完了
全候補生成関数定義完了


In [9]:
# 現状評価
CUTOFF = date(2020, 9, 15)
actuals = get_actuals(transactions, CUTOFF)
valid_customers = list(actuals.keys())
print(f"valid_customers: {len(valid_customers):,}")

print("\n【単体Recall@12】")
for name in candidate_files.keys():
    df = load_candidates(name, CUTOFF)
    if df is None:
        print(f"  {name:<25} ファイルなし")
        continue
    df_valid = df.filter(pl.col('customer_id').is_in(valid_customers))
    hits = []
    cand_dict = defaultdict(list)
    for row in df_valid.iter_rows(named=True):
        cand_dict[row['customer_id']].append(row['article_id'])
    
    for cid in valid_customers:
        actual_set = actuals[cid]
        cands = cand_dict.get(cid, [])[:12]
        hit = len(set(cands) & actual_set)
        hits.append(hit / len(actual_set) if actual_set else 0.0)
    
    print(f"  {name:<25} Recall@12: {np.mean(hits):.6f}")

print(f"\n【統合後Recall@12（best_weights）】")
print(f"  {best_recall:.6f}")

valid_customers: 68,984

【単体Recall@12】
  読み込み: /workspace/outputs/candidates/timedecay_2020-09-15.parquet
  timedecay                 Recall@12: 0.029429
  読み込み: /workspace/outputs/candidates/repurchase_2020-09-15.parquet
  repurchase                Recall@12: 0.019338
  読み込み: /workspace/outputs/candidates/variant_2020-09-15.parquet
  variant                   Recall@12: 0.012091
  読み込み: /workspace/outputs/candidates/also_bought_2020-09-15.parquet
  also_bought               Recall@12: 0.013404
  読み込み: /workspace/outputs/candidates/item2item_cf_2020-09-15.parquet
  item2item_cf              Recall@12: 0.011947
  読み込み: /workspace/outputs/candidates/category_popular_2020-09-15.parquet
  category_popular          Recall@12: 0.010244
  読み込み: /workspace/outputs/candidates/bin_popular_2020-09-15.parquet
  bin_popular               Recall@12: 0.019702
  読み込み: /workspace/outputs/candidates/general_pred_2020-09-15.parquet
  general_pred              Recall@12: 0.015016
  読み込み: /workspace/output

In [3]:
# # ============================================================
# # 全候補生成実行
# # ============================================================
# CUTOFF = date(2020, 9, 15)

# print(f"候補生成開始: cutoff={CUTOFF}")
# print("="*50)

# print("\n① timedecay...")
# td_df = build_timedecay(transactions, CUTOFF)

# print("\n② repurchase...")
# rp_df = build_repurchase(transactions, CUTOFF)

# print("\n③ variant...")
# var_df = build_variant(transactions, CUTOFF, td_df)

# print("\n④ also bought...")
# ab_df = build_also_bought(transactions, CUTOFF, rp_df)

# print("\n⑤ item2item CF...")
# cf_df = build_item2item_cf(transactions, CUTOFF, rp_df)

# print("\n⑥ category popular...")
# cat_df = build_category_popular(transactions, CUTOFF)

# print("\n⑦ bin popular...")
# bin_df = build_bin_popular(transactions, CUTOFF)

# print("\n⑧ general pred...")
# gen_df = build_general_pred(transactions, CUTOFF)

# print("\n⑨ purchase interval...")
# pi_df = build_purchase_interval(transactions, CUTOFF)

# print("\n⑩ user category popular...")
# ucp_df = build_user_category_popular(transactions, CUTOFF)

# print("\n" + "="*50)
# print("全候補生成完了!")
# print(f"保存先: {CANDIDATE_DIR}")

# # 各ロジックの候補数サマリー
# print("\n【候補数サマリー】")
# for name, df in [
#     ('timedecay', td_df),
#     ('repurchase', rp_df),
#     ('variant', var_df),
#     ('also_bought', ab_df),
#     ('item2item_cf', cf_df),
#     ('category_popular', cat_df),
#     ('bin_popular', bin_df),
#     ('general_pred', gen_df),
#     ('purchase_interval', pi_df),
#     ('user_category_popular', ucp_df),
# ]:
#     n_users = df['customer_id'].n_unique()
#     print(f"  {name:<25} 候補数: {len(df):>10,} / ユーザー数: {n_users:>8,}")

In [4]:
# print("\n⑥ category popular...")
# cat_df = build_category_popular(transactions, CUTOFF)

# print("\n⑦ bin popular...")
# bin_df = build_bin_popular(transactions, CUTOFF)

# print("\n⑧ general pred...")
# gen_df = build_general_pred(transactions, CUTOFF)

# print("\n⑨ purchase interval...")
# pi_df = build_purchase_interval(transactions, CUTOFF)

# print("\n⑩ user category popular...")
# ucp_df = build_user_category_popular(transactions, CUTOFF)

# print("\n" + "="*50)
# print("全候補生成完了!")

# print("\n【候補数サマリー】")
# for name, df in [
#     ('timedecay', td_df),
#     ('repurchase', rp_df),
#     ('variant', var_df),
#     ('also_bought', ab_df),
#     ('category_popular', cat_df),
#     ('bin_popular', bin_df),
#     ('general_pred', gen_df),
#     ('purchase_interval', pi_df),
#     ('user_category_popular', ucp_df),
# ]:
#     n_users = df['customer_id'].n_unique()
#     print(f"  {name:<25} 候補数: {len(df):>10,} / ユーザー数: {n_users:>8,}")

In [5]:
def build_item2item_cf(transactions, cutoff, top_n=20):
    """⑤item2item CF候補（高速版：事前に類似度辞書を作成）"""
    name = 'item2item_cf'
    cached = load_candidates(name, cutoff)
    if cached is not None:
        return cached
    
    print(f"  [{name}] 計算中...")
    from scipy.sparse import csr_matrix
    from sklearn.preprocessing import normalize
    
    df_train = transactions.filter(pl.col('t_dat') <= pl.lit(cutoff))
    
    user_item = (
        df_train
        .group_by(['customer_id', 'article_id'])
        .agg(pl.len().alias('count'))
    )
    
    n_users = len(customer_id_map)
    n_items = len(article_id_map)
    
    row = user_item['customer_id'].to_numpy()
    col = user_item['article_id'].to_numpy()
    data = user_item['count'].to_numpy().astype(np.float32)
    
    uim = csr_matrix((data, (row, col)), shape=(n_users, n_items))
    iim = uim.T.tocsr()  # item×user
    iim_norm = normalize(iim, norm='l2')
    
    print(f"    行列サイズ: {iim_norm.shape}")
    
    # 直近1ヶ月の購買商品のみ対象
    start_date = cutoff - timedelta(days=30)
    recent_articles = (
        df_train.filter(pl.col('t_dat') >= pl.lit(start_date))
        .select('article_id').unique()
        .to_series().to_list()
    )
    print(f"    直近1ヶ月の購買商品数: {len(recent_articles):,}")
    
    # 直近購買商品の類似商品を事前計算
    print("    類似度辞書を事前計算中...")
    similar_items_dict = {}
    for i, art in enumerate(recent_articles):
        if i % 10000 == 0:
            print(f"      進捗: {i:,} / {len(recent_articles):,}")
        if art >= iim_norm.shape[0]:
            continue
        sim_scores = iim_norm[art].dot(iim_norm.T).toarray().flatten()
        top_items = np.argsort(sim_scores)[::-1][1:top_n+1]
        similar_items_dict[art] = [
            (int(idx), float(sim_scores[idx]))
            for idx in top_items
            if sim_scores[idx] > 0
        ]
    
    print(f"    類似度辞書完成: {len(similar_items_dict):,}商品")
    
    # ユーザーごとに候補生成
    print("    ユーザーごとに候補生成中...")
    recent_hist = (
        df_train.filter(pl.col('t_dat') >= pl.lit(start_date))
        .group_by('customer_id')
        .agg(pl.col('article_id').unique().alias('articles'))
    )
    
    rows_list = []
    for i, row_data in enumerate(recent_hist.iter_rows(named=True)):
        if i % 100000 == 0:
            print(f"      進捗: {i:,} / {len(recent_hist):,}")
        
        cid = row_data['customer_id']
        hist_arts = row_data['articles']
        purchased_set = set(hist_arts)
        
        candidates = {}
        for art in hist_arts:
            for sim_art, sim_score in similar_items_dict.get(art, []):
                if sim_art not in purchased_set:
                    candidates[sim_art] = candidates.get(sim_art, 0) + sim_score
        
        sorted_cands = sorted(candidates.items(), key=lambda x: -x[1])[:top_n]
        for rank, (art, score) in enumerate(sorted_cands, start=1):
            rows_list.append({
                'customer_id': cid,
                'article_id': art,
                'score': score,
                'rank': rank,
            })
    
    result = pl.DataFrame(rows_list).with_columns([
        pl.col('customer_id').cast(pl.Int32),
        pl.col('article_id').cast(pl.Int32),
        pl.col('rank').cast(pl.Int32),
    ])
    print(f"  [{name}] 候補数: {len(result):,}")
    return save_candidates(result, name, cutoff)

print("関数定義完了")

関数定義完了


In [6]:
# print("\n⑤ item2item CF...")
# cf_df = build_item2item_cf(transactions, CUTOFF)
# print(f"  ユーザー数: {cf_df['customer_id'].n_unique():,}")

In [6]:
# ============================================================
# ⑪ グリッドサーチ
# ============================================================
# from itertools import product as iterproduct

# CUTOFF = date(2020, 9, 15)
# VALID_CUTOFF = date(2020, 9, 15)
# K = 12

# # 候補ファイル読み込み
# print("候補ファイル読み込み中...")
candidate_files = {
    'timedecay':           load_candidates('timedecay', CUTOFF),
    'repurchase':          load_candidates('repurchase', CUTOFF),
    'variant':             load_candidates('variant', CUTOFF),
    'also_bought':         load_candidates('also_bought', CUTOFF),
    'item2item_cf':        load_candidates('item2item_cf', CUTOFF),
    'category_popular':    load_candidates('category_popular', CUTOFF),
    'bin_popular':         load_candidates('bin_popular', CUTOFF),
    'general_pred':        load_candidates('general_pred', CUTOFF),
    'purchase_interval':   load_candidates('purchase_interval', CUTOFF),
    'user_category_popular': load_candidates('user_category_popular', CUTOFF),
}

# # valid_yを取得
# actuals = get_actuals(transactions, VALID_CUTOFF)
# valid_customers = list(actuals.keys())
# print(f"valid_customers: {len(valid_customers):,}")

# # 各ロジックをユーザーごとのdictに変換（高速化）
# print("候補dictに変換中...")
# cand_dicts = {}
# for name, df in candidate_files.items():
#     if df is None:
#         print(f"  {name}: スキップ（ファイルなし）")
#         continue
#     # valid_customersのみに絞る
#     df_valid = df.filter(pl.col('customer_id').is_in(valid_customers))
#     d = defaultdict(list)
#     for row in df_valid.iter_rows(named=True):
#         d[row['customer_id']].append((row['article_id'], row['score'], row['rank']))
#     cand_dicts[name] = dict(d)
#     print(f"  {name}: {len(d):,}ユーザー")

# # スコアを正規化（0〜1に変換）
# print("スコア正規化中...")
# def normalize_scores(cand_dict):
#     """各ユーザーのスコアを0〜1に正規化"""
#     normalized = {}
#     for cid, cands in cand_dict.items():
#         if not cands:
#             continue
#         max_score = max(s for _, s, _ in cands)
#         min_score = min(s for _, s, _ in cands)
#         diff = max_score - min_score
#         if diff == 0:
#             normalized[cid] = [(art, 1.0, rank) for art, s, rank in cands]
#         else:
#             normalized[cid] = [(art, (s - min_score) / diff, rank) for art, s, rank in cands]
#     return normalized

# norm_cand_dicts = {name: normalize_scores(d) for name, d in cand_dicts.items()}

# def compute_recall_with_weights(weights, norm_cand_dicts, actuals, valid_customers, k=12):
#     """重みを指定してRecall@kを計算"""
#     hits = []
#     for cid in valid_customers:
#         actual_set = actuals[cid]
        
#         # 各ロジックのスコアを重み付きで合算
#         score_dict = {}
#         for name, w in weights.items():
#             if w == 0 or name not in norm_cand_dicts:
#                 continue
#             for art, score, rank in norm_cand_dicts[name].get(cid, []):
#                 score_dict[art] = score_dict.get(art, 0) + w * score
        
#         # スコア上位k件
#         top_k = sorted(score_dict.items(), key=lambda x: -x[1])[:k]
#         top_k_arts = set(art for art, _ in top_k)
        
#         hit = len(top_k_arts & actual_set)
#         hits.append(hit / len(actual_set) if actual_set else 0.0)
    
#     return np.mean(hits)

# # グリッドサーチ
# print("\nグリッドサーチ開始...")
# weight_candidates = [0.0, 0.5, 1.0, 2.0]
# logic_names = list(norm_cand_dicts.keys())

# # まず各ロジック単体のRecall@12を確認
# print("\n【単体Recall@12】")
# for name in logic_names:
#     weights = {n: 0.0 for n in logic_names}
#     weights[name] = 1.0
#     recall = compute_recall_with_weights(weights, norm_cand_dicts, actuals, valid_customers)
#     print(f"  {name:<25} Recall@12: {recall:.6f}")

# # 重要なロジックに絞ってグリッドサーチ
# # まず上位ロジックを特定してから探索
# print("\n【グリッドサーチ】")
# print("上位4ロジックで探索します...")

# # 単体Recallが高い上位4つを選択して探索
# top_logics = ['timedecay', 'repurchase', 'bin_popular', 'general_pred']
# other_logics = [n for n in logic_names if n not in top_logics]

# best_recall = 0
# best_weights = None
# results_gs = []

# total = len(weight_candidates) ** len(top_logics)
# print(f"組み合わせ数: {total:,}")

# for i, combo in enumerate(iterproduct(weight_candidates, repeat=len(top_logics))):
#     weights = {n: 1.0 for n in other_logics}  # その他は固定1.0
#     weights.update(dict(zip(top_logics, combo)))
    
#     if all(w == 0 for w in combo):
#         continue
    
#     recall = compute_recall_with_weights(weights, norm_cand_dicts, actuals, valid_customers)
#     results_gs.append((recall, weights.copy()))
    
#     if recall > best_recall:
#         best_recall = recall
#         best_weights = weights.copy()
#         print(f"  [{i+1}/{total}] 新ベスト Recall@12: {best_recall:.6f} weights: {dict(zip(top_logics, combo))}")

# print(f"\nグリッドサーチ完了!")
# print(f"最良Recall@12: {best_recall:.6f}")
# print(f"最良重み: {best_weights}")

  読み込み: /workspace/outputs/candidates/timedecay_2020-09-15.parquet
  読み込み: /workspace/outputs/candidates/repurchase_2020-09-15.parquet
  読み込み: /workspace/outputs/candidates/variant_2020-09-15.parquet
  読み込み: /workspace/outputs/candidates/also_bought_2020-09-15.parquet
  読み込み: /workspace/outputs/candidates/item2item_cf_2020-09-15.parquet
  読み込み: /workspace/outputs/candidates/category_popular_2020-09-15.parquet
  読み込み: /workspace/outputs/candidates/bin_popular_2020-09-15.parquet
  読み込み: /workspace/outputs/candidates/general_pred_2020-09-15.parquet
  読み込み: /workspace/outputs/candidates/purchase_interval_2020-09-15.parquet
  読み込み: /workspace/outputs/candidates/user_category_popular_2020-09-15.parquet


In [8]:
import json

# best_weightsを読み込み
weights_path = Path('/workspace/outputs/candidates/best_weights.json')
with open(weights_path, 'r') as f:
    best_weights = json.load(f)
print(f"best_weights: {best_weights}")

# 候補ファイル読み込み
print("\n候補ファイル読み込み中...")
CUTOFF = date(2020, 9, 15)
candidate_names = [
    'timedecay', 'repurchase', 'variant', 'also_bought',
    'item2item_cf', 'category_popular', 'bin_popular',
    'general_pred', 'purchase_interval', 'user_category_popular'
]

candidate_files = {}
for name in candidate_names:
    df = load_candidates(name, CUTOFF)
    if df is not None:
        candidate_files[name] = df
        print(f"  {name}: {len(df):,}行")
    else:
        print(f"  {name}: ファイルなし（スキップ）")

# valid_yを取得
VALID_CUTOFF = date(2020, 9, 15)
actuals = get_actuals(transactions, VALID_CUTOFF)
valid_customers = list(actuals.keys())
print(f"\nvalid_customers: {len(valid_customers):,}")

# 各ロジックをdictに変換（valid_customersのみ）
print("\n候補dictに変換中...")
cand_dicts = {}
for name, df in candidate_files.items():
    df_valid = df.filter(pl.col('customer_id').is_in(valid_customers))
    d = defaultdict(list)
    for row in df_valid.iter_rows(named=True):
        d[row['customer_id']].append((row['article_id'], row['score'], row['rank']))
    cand_dicts[name] = dict(d)
    print(f"  {name}: {len(d):,}ユーザー")

# スコア正規化
print("\nスコア正規化中...")
def normalize_scores(cand_dict):
    normalized = {}
    for cid, cands in cand_dict.items():
        if not cands:
            continue
        max_score = max(s for _, s, _ in cands)
        min_score = min(s for _, s, _ in cands)
        diff = max_score - min_score
        if diff == 0:
            normalized[cid] = [(art, 1.0, rank) for art, s, rank in cands]
        else:
            normalized[cid] = [(art, (s - min_score) / diff, rank) for art, s, rank in cands]
    return normalized

# best_weightsに含まれるロジックのみ正規化
norm_cand_dicts = {}
for name, d in cand_dicts.items():
    if name in best_weights and best_weights[name] > 0:
        norm_cand_dicts[name] = normalize_scores(d)
        print(f"  {name}: 正規化完了（重み={best_weights[name]}）")
    else:
        print(f"  {name}: スキップ（重み=0）")

print("\n完了!")

best_weights: {'variant': 1.0, 'also_bought': 1.0, 'item2item_cf': 1.0, 'category_popular': 1.0, 'purchase_interval': 1.0, 'user_category_popular': 1.0, 'timedecay': 2.0, 'repurchase': 1.0, 'bin_popular': 0.5, 'general_pred': 0.0}

候補ファイル読み込み中...
  読み込み: /workspace/outputs/candidates/timedecay_2020-09-15.parquet
  timedecay: 10,397,543行
  読み込み: /workspace/outputs/candidates/repurchase_2020-09-15.parquet
  repurchase: 25,646,956行
  読み込み: /workspace/outputs/candidates/variant_2020-09-15.parquet
  variant: 23,426,226行
  読み込み: /workspace/outputs/candidates/also_bought_2020-09-15.parquet
  also_bought: 4,426,368行
  読み込み: /workspace/outputs/candidates/item2item_cf_2020-09-15.parquet
  item2item_cf: 4,989,943行
  読み込み: /workspace/outputs/candidates/category_popular_2020-09-15.parquet
  category_popular: 24,731,330行
  読み込み: /workspace/outputs/candidates/bin_popular_2020-09-15.parquet
  bin_popular: 68,597,889行
  読み込み: /workspace/outputs/candidates/general_pred_2020-09-15.parquet
  general_pred:

In [9]:
import json
from pathlib import Path

best_weights = {
    'variant': 1.0,
    'also_bought': 1.0,
    'item2item_cf': 1.0,
    'category_popular': 1.0,
    'purchase_interval': 1.0,
    'user_category_popular': 1.0,
    'timedecay': 2.0,
    'repurchase': 1.0,
    'bin_popular': 0.5,
    'general_pred': 0.0
}

weights_path = Path('/workspace/outputs/candidates/best_weights.json')
weights_path.parent.mkdir(parents=True, exist_ok=True)
with open(weights_path, 'w') as f:
    json.dump(best_weights, f, indent=2)
print(f"保存完了: {weights_path}")
print(f"best_recall: 0.056413")
print(f"best_weights: {best_weights}")

保存完了: /workspace/outputs/candidates/best_weights.json
best_recall: 0.056413
best_weights: {'variant': 1.0, 'also_bought': 1.0, 'item2item_cf': 1.0, 'category_popular': 1.0, 'purchase_interval': 1.0, 'user_category_popular': 1.0, 'timedecay': 2.0, 'repurchase': 1.0, 'bin_popular': 0.5, 'general_pred': 0.0}


In [10]:
# ============================================================
# ⑫ 最適重みで候補生成
# ============================================================
N_CANDIDATES = 100  # リランキング用に多めに候補を取得

print("最適重みで候補生成中...")

def generate_candidates_with_weights(
    customer_ids, cand_dicts, weights, n_candidates=100
):
    """重み付きスコアで候補を生成"""
    rows = []
    for cid in customer_ids:
        score_dict = {}
        for name, w in weights.items():
            if w == 0 or name not in cand_dicts:
                continue
            for art, score, rank in cand_dicts[name].get(cid, []):
                score_dict[art] = score_dict.get(art, 0) + w * score
        
        sorted_cands = sorted(score_dict.items(), key=lambda x: -x[1])[:n_candidates]
        for rank, (art, score) in enumerate(sorted_cands, start=1):
            rows.append({
                'customer_id': cid,
                'article_id': art,
                'score': score,
                'rank': rank,
            })
    return rows

# valid用候補生成（リランキング学習用）
print("valid候補生成中...")
valid_rows = generate_candidates_with_weights(
    valid_customers, norm_cand_dicts, best_weights, N_CANDIDATES
)
valid_cands_df = pl.DataFrame(valid_rows).with_columns([
    pl.col('customer_id').cast(pl.Int32),
    pl.col('article_id').cast(pl.Int32),
    pl.col('rank').cast(pl.Int32),
])
print(f"valid候補数: {len(valid_cands_df):,}")

# Recall確認
actuals_valid = get_actuals(transactions, VALID_CUTOFF)
hit_count = 0
total_actual = 0
for cid in valid_customers:
    actual_set = actuals_valid.get(cid, set())
    if not actual_set:
        continue
    cand_arts = set(
        valid_cands_df.filter(pl.col('customer_id') == cid)['article_id'].to_list()
    )
    hit_count += len(cand_arts & actual_set)
    total_actual += len(actual_set)

print(f"候補生成後Recall@{N_CANDIDATES}: {hit_count/total_actual:.6f}")

最適重みで候補生成中...
valid候補生成中...
valid候補数: 6,392,031
候補生成後Recall@100: 0.111469


In [ ]:
# ============================================================
# ⑬ リランキング（CatBoost）1fold
# ============================================================
from catboost import CatBoost, Pool
import pandas as pd

feature_cols = [
    'rank', 'score', 'is_history', 'is_variant',
    'age', 'days_since_last_purchase', 'repeat_count',
    'product_type_no', 'garment_group_no',
    'week0_sales', 'week1_sales',
    'age_bin_purchase_rate', 'days_since_last_product_type',
]

def create_features_light(transactions, customers, articles, cutoff, candidates_df):
    """軽量特徴量作成"""
    print("  特徴量作成中...")
    df_train = transactions.filter(pl.col('t_dat') <= pl.lit(cutoff))
    
    # is_history（timedecayとのjoin）
    td_df = load_candidates('timedecay', cutoff).select(['customer_id', 'article_id']).with_columns(
        pl.lit(1).alias('is_history')
    )
    # is_variant（variantとのjoin）
    var_df = load_candidates('variant', cutoff).select(['customer_id', 'article_id']).with_columns(
        pl.lit(1).alias('is_variant')
    )
    
    # repeat_count
    repeat_count = (
        df_train.group_by(['customer_id', 'article_id'])
        .agg(pl.len().alias('repeat_count'))
    )
    
    # days_since_last_purchase
    user_last = (
        df_train.group_by('customer_id')
        .agg(pl.col('t_dat').max().alias('last_date'))
    ).with_columns(
        (pl.lit(cutoff) - pl.col('last_date')).dt.total_days().alias('days_since_last_purchase')
    ).select(['customer_id', 'days_since_last_purchase'])
    
    # article属性
    article_attrs = articles.select(['article_id', 'product_type_no', 'garment_group_no'])
    
    # 週別販売数
    week0_sales = (
        df_train.filter(pl.col('t_dat') > pl.lit(cutoff - timedelta(days=7)))
        .group_by('article_id').agg(pl.len().alias('week0_sales'))
    )
    week1_sales = (
        df_train.filter(
            (pl.col('t_dat') > pl.lit(cutoff - timedelta(days=14))) &
            (pl.col('t_dat') <= pl.lit(cutoff - timedelta(days=7)))
        )
        .group_by('article_id').agg(pl.len().alias('week1_sales'))
    )
    
    # 年齢ビン別購買率
    age_bin_counts = (
        df_train.join(customers.select(['customer_id', 'age_bin']), on='customer_id', how='left')
        .group_by(['age_bin', 'article_id']).agg(pl.len().alias('age_bin_article_count'))
    )
    age_bin_totals = (
        df_train.join(customers.select(['customer_id', 'age_bin']), on='customer_id', how='left')
        .group_by('age_bin').agg(pl.len().alias('age_bin_total'))
    )
    age_bin_purchase_rate = (
        age_bin_counts.join(age_bin_totals, on='age_bin', how='left')
        .with_columns((pl.col('age_bin_article_count') / pl.col('age_bin_total')).alias('age_bin_purchase_rate'))
        .select(['age_bin', 'article_id', 'age_bin_purchase_rate'])
    )
    
    # product_type別最終購買日
    tx_with_attrs = df_train.join(
        articles.select(['article_id', 'product_type_no']), on='article_id', how='left'
    )
    last_purchase_by_product_type = (
        tx_with_attrs.group_by(['customer_id', 'product_type_no'])
        .agg(pl.col('t_dat').max().alias('last_purchase_product_type'))
    )
    
    # 結合
    result = candidates_df.clone()
    result = result.join(td_df, on=['customer_id', 'article_id'], how='left').with_columns(
        pl.col('is_history').fill_null(0)
    )
    result = result.join(var_df, on=['customer_id', 'article_id'], how='left').with_columns(
        pl.col('is_variant').fill_null(0)
    )
    result = result.join(article_attrs, on='article_id', how='left')
    result = result.join(customers.select(['customer_id', 'age', 'age_bin']), on='customer_id', how='left')
    result = result.join(user_last, on='customer_id', how='left')
    result = result.join(repeat_count, on=['customer_id', 'article_id'], how='left').with_columns(
        pl.col('repeat_count').fill_null(0)
    )
    result = result.join(week0_sales, on='article_id', how='left').with_columns(pl.col('week0_sales').fill_null(0))
    result = result.join(week1_sales, on='article_id', how='left').with_columns(pl.col('week1_sales').fill_null(0))
    result = result.join(age_bin_purchase_rate, on=['age_bin', 'article_id'], how='left').with_columns(
        pl.col('age_bin_purchase_rate').fill_null(0)
    )
    result = result.join(last_purchase_by_product_type, on=['customer_id', 'product_type_no'], how='left').with_columns(
        pl.when(pl.col('last_purchase_product_type').is_not_null())
        .then((pl.lit(cutoff) - pl.col('last_purchase_product_type')).dt.total_days())
        .otherwise(-1).alias('days_since_last_product_type')
    ).drop('last_purchase_product_type')
    
    result = result.with_columns([
        pl.col('age').fill_null(-1),
        pl.col('days_since_last_purchase').fill_null(-1),
        pl.col('product_type_no').fill_null(-1),
        pl.col('garment_group_no').fill_null(-1),
        pl.col('days_since_last_product_type').fill_null(-1),
    ])
    
    print(f"  特徴量作成完了: {result.shape}")
    return result

# 学習データ作成（1fold: cutoff=2020-09-08）
TRAIN_CUTOFF = date(2020, 9, 8)
print(f"\n学習データ作成中（cutoff={TRAIN_CUTOFF}）...")

# train用候補生成
train_cutoff_actuals = get_actuals(transactions, TRAIN_CUTOFF)
train_customers = list(train_cutoff_actuals.keys())

# train用候補ファイル読み込み（なければ生成）
print("train用候補ファイル確認中...")
train_cand_files = {}
for name in candidate_files.keys():
    cached = load_candidates(name, TRAIN_CUTOFF)
    if cached is None:
        print(f"  {name}: 生成中...")
        if name == 'timedecay':
            cached = build_timedecay(transactions, TRAIN_CUTOFF)
        elif name == 'repurchase':
            cached = build_repurchase(transactions, TRAIN_CUTOFF)
        elif name == 'variant':
            td = build_timedecay(transactions, TRAIN_CUTOFF)
            cached = build_variant(transactions, TRAIN_CUTOFF, td)
        elif name == 'also_bought':
            cached = build_also_bought(transactions, TRAIN_CUTOFF, None)
        elif name == 'item2item_cf':
            cached = build_item2item_cf(transactions, TRAIN_CUTOFF)
        elif name == 'category_popular':
            cached = build_category_popular(transactions, TRAIN_CUTOFF)
        elif name == 'bin_popular':
            cached = build_bin_popular(transactions, TRAIN_CUTOFF)
        elif name == 'general_pred':
            cached = build_general_pred(transactions, TRAIN_CUTOFF)
        elif name == 'purchase_interval':
            cached = build_purchase_interval(transactions, TRAIN_CUTOFF)
        elif name == 'user_category_popular':
            cached = build_user_category_popular(transactions, TRAIN_CUTOFF)
    train_cand_files[name] = cached

# train用候補をdictに変換
train_cand_dicts = {}
for name, df in train_cand_files.items():
    if df is None:
        continue
    df_valid = df.filter(pl.col('customer_id').is_in(train_customers))
    d = defaultdict(list)
    for row in df_valid.iter_rows(named=True):
        d[row['customer_id']].append((row['article_id'], row['score'], row['rank']))
    train_cand_dicts[name] = dict(d)

# train用スコア正規化
norm_train_cand_dicts = {name: normalize_scores(d) for name, d in train_cand_dicts.items()}

# train候補生成
train_rows = generate_candidates_with_weights(
    train_customers, norm_train_cand_dicts, best_weights, N_CANDIDATES
)
train_cands_df = pl.DataFrame(train_rows).with_columns([
    pl.col('customer_id').cast(pl.Int32),
    pl.col('article_id').cast(pl.Int32),
    pl.col('rank').cast(pl.Int32),
])

# labelを付与
actuals_train_df = pl.DataFrame([
    {'customer_id': int(cid), 'article_id': int(aid), 'label': 1}
    for cid, aids in train_cutoff_actuals.items()
    for aid in aids
]).with_columns([
    pl.col('customer_id').cast(pl.Int32),
    pl.col('article_id').cast(pl.Int32),
])
train_cands_df = train_cands_df.join(actuals_train_df, on=['customer_id', 'article_id'], how='left').with_columns(
    pl.col('label').fill_null(0)
)

# 特徴量作成
train_df = create_features_light(transactions, customers, articles, TRAIN_CUTOFF, train_cands_df).to_pandas()
print(f"学習データ: {len(train_df):,}行 / 正例: {train_df['label'].sum():,}")

# CatBoost学習
print("\nCatBoost学習中...")
train_df_sorted = train_df.sort_values('customer_id')
train_pool = Pool(
    data=train_df_sorted[feature_cols],
    label=train_df_sorted['label'],
    group_id=train_df_sorted['customer_id']
)
model = CatBoost({
    'loss_function': 'YetiRank',
    'iterations': 200,
    'learning_rate': 0.05,
    'depth': 6,
    'verbose': 20,
    'early_stopping_rounds': 20,
})
model.fit(train_pool)
print("学習完了!")

# valid評価
print("\nvalid評価中...")
actuals_valid_label = pl.DataFrame([
    {'customer_id': int(cid), 'article_id': int(aid), 'label': 1}
    for cid, aids in actuals_valid.items()
    for aid in aids
]).with_columns([
    pl.col('customer_id').cast(pl.Int32),
    pl.col('article_id').cast(pl.Int32),
])
valid_cands_df = valid_cands_df.join(
    actuals_valid_label, on=['customer_id', 'article_id'], how='left'
).with_columns(pl.col('label').fill_null(0))

valid_df = create_features_light(transactions, customers, articles, VALID_CUTOFF, valid_cands_df).to_pandas()
valid_df['pred_score'] = model.predict(valid_df[feature_cols])

metric_acc = {k: {'customer_recall': [], 'map': []} for k in [12, 24, 50, 100]}
for cid, group in valid_df.groupby('customer_id'):
    actual_set = actuals_valid.get(cid, set())
    if not actual_set:
        continue
    prediction = group.sort_values('pred_score', ascending=False)['article_id'].tolist()
    for k in [12, 24, 50, 100]:
        hits = len(set(prediction[:k]) & actual_set)
        metric_acc[k]['customer_recall'].append(hits / len(actual_set))
        metric_acc[k]['map'].append(apk(actual_set, prediction, k))

print("\n【valid評価】")
print(f"{'k':>4} {'recall':>10} {'MAP':>10}")
print("-"*30)
for k in [12, 24, 50, 100]:
    recall = np.mean(metric_acc[k]['customer_recall'])
    map_k = np.mean(metric_acc[k]['map'])
    print(f"{k:>4} {recall:>10.6f} {map_k:>10.6f}")


学習データ作成中（cutoff=2020-09-08）...
train用候補ファイル確認中...
  読み込み: /workspace/outputs/candidates/timedecay_2020-09-08.parquet
  読み込み: /workspace/outputs/candidates/repurchase_2020-09-08.parquet
  読み込み: /workspace/outputs/candidates/variant_2020-09-08.parquet
  読み込み: /workspace/outputs/candidates/also_bought_2020-09-08.parquet
  読み込み: /workspace/outputs/candidates/item2item_cf_2020-09-08.parquet
  読み込み: /workspace/outputs/candidates/category_popular_2020-09-08.parquet
  読み込み: /workspace/outputs/candidates/bin_popular_2020-09-08.parquet
  読み込み: /workspace/outputs/candidates/general_pred_2020-09-08.parquet
  読み込み: /workspace/outputs/candidates/purchase_interval_2020-09-08.parquet
  読み込み: /workspace/outputs/candidates/user_category_popular_2020-09-08.parquet
  特徴量作成中...
  読み込み: /workspace/outputs/candidates/timedecay_2020-09-08.parquet
  読み込み: /workspace/outputs/candidates/variant_2020-09-08.parquet
  特徴量作成完了: (6698277, 17)
学習データ: 6,698,277行 / 正例: 24,481

CatBoost学習中...
0:	total: 869ms	remaining: 

: 

In [ ]:
# ============================================================
# ⑭ 提出ファイル作成
# ============================================================
from tqdm import tqdm

OUTPUT_DIR = Path('/workspace/outputs/submissions')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SUB_CUTOFF = transactions['t_dat'].max()
print(f"提出用cutoff: {SUB_CUTOFF}")

# 提出用候補ファイル読み込み（なければ生成）
print("提出用候補ファイル確認中...")
sub_cand_files = {}
for name in candidate_files.keys():
    cached = load_candidates(name, SUB_CUTOFF)
    if cached is None:
        print(f"  {name}: 生成中...")
        if name == 'timedecay':
            cached = build_timedecay(transactions, SUB_CUTOFF)
        elif name == 'repurchase':
            cached = build_repurchase(transactions, SUB_CUTOFF)
        elif name == 'variant':
            td = build_timedecay(transactions, SUB_CUTOFF)
            cached = build_variant(transactions, SUB_CUTOFF, td)
        elif name == 'also_bought':
            cached = build_also_bought(transactions, SUB_CUTOFF, None)
        elif name == 'item2item_cf':
            cached = build_item2item_cf(transactions, SUB_CUTOFF)
        elif name == 'category_popular':
            cached = build_category_popular(transactions, SUB_CUTOFF)
        elif name == 'bin_popular':
            cached = build_bin_popular(transactions, SUB_CUTOFF)
        elif name == 'general_pred':
            cached = build_general_pred(transactions, SUB_CUTOFF)
        elif name == 'purchase_interval':
            cached = build_purchase_interval(transactions, SUB_CUTOFF)
        elif name == 'user_category_popular':
            cached = build_user_category_popular(transactions, SUB_CUTOFF)
    sub_cand_files[name] = cached

# 提出用候補をdictに変換（全ユーザー）
print("提出用候補dictに変換中...")
all_customer_ids_list = list(customer_id_map.values())

sub_cand_dicts = {}
for name, df in sub_cand_files.items():
    if df is None:
        continue
    d = defaultdict(list)
    for row in df.iter_rows(named=True):
        d[row['customer_id']].append((row['article_id'], row['score'], row['rank']))
    sub_cand_dicts[name] = dict(d)

norm_sub_cand_dicts = {name: normalize_scores(d) for name, d in sub_cand_dicts.items()}

# 全ユーザーの候補生成（バッチ処理）
print("全ユーザーの候補生成中...")
BATCH_SIZE = 100000
all_rows = []

for batch_start in tqdm(range(0, len(all_customer_ids_list), BATCH_SIZE), desc="候補生成"):
    batch = all_customer_ids_list[batch_start:batch_start+BATCH_SIZE]
    batch_rows = generate_candidates_with_weights(
        batch, norm_sub_cand_dicts, best_weights, N_CANDIDATES
    )
    all_rows.extend(batch_rows)

all_cands_df = pl.DataFrame(all_rows).with_columns([
    pl.col('customer_id').cast(pl.Int32),
    pl.col('article_id').cast(pl.Int32),
    pl.col('rank').cast(pl.Int32),
])
print(f"全候補数: {len(all_cands_df):,}")

# 特徴量作成
print("特徴量作成中...")
all_features_df = create_features_light(
    transactions, customers, articles, SUB_CUTOFF, all_cands_df
).to_pandas()

# 予測
print("予測中...")
all_features_df['pred_score'] = model.predict(all_features_df[feature_cols])

# 上位12件を選択
print("上位12件選択中...")
submission = (
    all_features_df
    .sort_values(['customer_id', 'pred_score'], ascending=[True, False])
    .groupby('customer_id')
    .head(12)
    .groupby('customer_id')['article_id']
    .apply(lambda x: ' '.join([article_id_reverse[i] for i in x]))
    .reset_index()
    .rename(columns={'article_id': 'prediction'})
)

# customer_idを元の文字列に戻す
submission['customer_id'] = submission['customer_id'].map(customer_id_reverse)

# general_predのfallback
general_pred_top12 = ' '.join([
    article_id_reverse[i] for i in 
    load_candidates('general_pred', SUB_CUTOFF)
    .sort('score', descending=True)
    .select('article_id').head(12)
    .to_series().to_list()
])

all_customers_final = pd.DataFrame({'customer_id': list(customer_id_map.keys())})
all_customers_final = all_customers_final.merge(submission, on='customer_id', how='left')
all_customers_final['prediction'] = all_customers_final['prediction'].fillna(general_pred_top12)

# 保存
all_customers_final.to_csv(OUTPUT_DIR / 'submission_rerank_v5.csv', index=False)
print(f"\n保存完了: submission_rerank_v5.csv")
print(f"ユーザー数: {len(all_customers_final):,}")
print(all_customers_final.head(5))

提出用cutoff: 2020-09-22
提出用候補ファイル確認中...
  読み込み: /workspace/outputs/candidates/timedecay_2020-09-22.parquet
  読み込み: /workspace/outputs/candidates/repurchase_2020-09-22.parquet
  variant: 生成中...
  読み込み: /workspace/outputs/candidates/timedecay_2020-09-22.parquet
  [variant] 計算中...
